# 04 · H1: does the EEG change as awakening approaches?

**H1:** within participants, posterior delta power and the aperiodic exponent fall from E2 to E4.
Linear mixed model per measure over E2-E4: time (epoch centre, per 30 s) + site + artifact rate,
random intercepts for participant and for awakening within participant. Holm across the two primary tests.

**RQ3 (exploratory):** does the slope depend on report richness (log word count of the transcription)?

Everything below the primary section is exploratory and labelled so.

In [ ]:
# Bootstrap: same cell in every notebook. On Colab: mount Drive, get the repo, install extras.
# The repo is private: add a read-only GitHub token to Colab Secrets as GH_TOKEN.
import os, sys, subprocess, base64
REPO_URL = "https://github.com/sanjanasinghx/dream2image-es606.git"
if "google.colab" in sys.modules and not os.environ.get("D2I_HEADLESS"):  # headless = run by d2i.runner
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    REPO = "/content/dream2image-es606"
    try:
        auth = base64.b64encode(f"x-access-token:{userdata.get('GH_TOKEN')}".encode()).decode()
        git = ["git", "-c", f"http.extraHeader=Authorization: Basic {auth}"]  # token never written to disk
    except Exception:
        git = ["git"]  # no secret: works once the repo is public
    if not os.path.exists(REPO):
        subprocess.run([*git, "clone", "-q", REPO_URL, REPO], check=True)
    else:
        subprocess.run([*git, "-C", REPO, "fetch", "-q", "origin"], check=True)
        subprocess.run(["git", "-C", REPO, "reset", "-q", "--hard", "origin/main"], check=True)
    freeze = lambda: subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
    before = freeze()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-colab.txt"], check=True)
    if freeze() != before:  # e.g. numpy upgraded under an already-loaded copy -> broken imports
        print("Packages changed: restarting the runtime. When it reconnects, Run all again.")
        os.kill(os.getpid(), 9)
else:
    REPO = os.path.abspath("..")
sys.path.insert(0, REPO)
from d2i import env
cfg = env.load_config()
rng = env.seed_everything(cfg["seed"])
print("data root:", env.data_root())

In [ ]:
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from d2i import data, stats
hc = cfg["h1"]
der, tables, figs = env.path(cfg, "derived"), env.path(cfg, "results") / "tables", env.path(cfg, "results") / "figures"
tables.mkdir(exist_ok=True); figs.mkdir(exist_ok=True)
words = {m["sample_id"]: len((m.get("transcription") or "").split()) for m in data.load_meta(cfg)}
def prep(df):
    d = df[df.epoch.isin(hc["epochs"])].copy()
    d["time30"] = d.center_s / 30.0
    d["richness_z"] = np.log(d.sample_id.map(words))
    d["richness_z"] = (d.richness_z - d.richness_z.mean()) / d.richness_z.std()
    return d
roi = prep(pd.read_csv(der / "features_roi.csv"))
long = prep(pd.read_csv(der / "features_long.csv"))
long = long[long.ok]
print(f"ROI rows {len(roi)}, awakenings {roi.sample_id.nunique()}, participants {roi.participant.nunique()}")

## 1. Primary test (confirmatory)

In [ ]:
rows = []
for y in hc["primary"]:
    rows.append(stats.fit_trend(roi, y, hc["covariates"]))
    rows.append(stats.fit_trend(roi, y, [c for c in hc["covariates"] if c != "reject_rate"]))
prim = pd.DataFrame(rows)
main = prim.covariates == "+".join(hc["covariates"])
prim.loc[main, "p_holm"] = stats.holm(prim.loc[main, "p"])
prim.to_csv(tables / "h1_primary.csv", index=False)
print(prim[["y", "covariates", "estimate", "ci_low", "ci_high", "std_estimate", "p", "p_holm",
            "n_awakenings", "n_participants", "converged"]].round(4).to_string(index=False))

## 2. RQ3: does the slope depend on report richness? (exploratory)

In [ ]:
rq3 = pd.DataFrame([stats.fit_trend(roi, y, hc["covariates"], interaction="richness_z") for y in hc["primary"]])
rq3.to_csv(tables / "rq3_richness.csv", index=False)
print("word count: median", int(np.median(list(words.values()))), "range", min(words.values()), "-", max(words.values()))
print(rq3[["y", "term", "estimate", "ci_low", "ci_high", "p", "converged"]].round(4).to_string(index=False))

## 3. All bands, posterior ROI, and the channel map (exploratory, FDR)

In [ ]:
measures = [c for c in roi.columns if c.startswith("logpow_")] + ["ap_exponent", "ap_offset"]
expl = pd.DataFrame([stats.fit_trend(roi, y, hc["covariates"]) for y in measures])
expl["p_fdr"] = stats.fdr_bh(expl.p)
expl.to_csv(tables / "h1_exploratory_roi.csv", index=False)
print(expl[["y", "std_estimate", "ci_low", "ci_high", "p", "p_fdr"]].round(4).to_string(index=False))

chan = []
for ch, g in long.groupby("channel"):
    for y in ["logpow_delta", "logpow_alpha", "logpow_beta", "ap_exponent"]:
        chan.append({"channel": ch, **stats.fit_trend(g, y, hc["covariates"])})
chan = pd.DataFrame(chan)
chan["p_fdr"] = stats.fdr_bh(chan.p)
chan.to_csv(tables / "h1_exploratory_channels.csv", index=False)
print("channel x measure tests surviving FDR:", int((chan.p_fdr < hc["alpha"]).sum()), "of", len(chan))

## 4. Figure

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
labels = {"logpow_delta": "posterior log10 delta power", "ap_exponent": "posterior aperiodic exponent"}
for a, y in zip(ax[:2], hc["primary"]):
    d = roi.dropna(subset=[y]).copy()
    d["centered"] = d[y] - d.groupby("sample_id")[y].transform("mean")  # within-awakening change
    m = d.groupby("center_s").centered.agg(["mean", "sem"])
    a.errorbar(m.index, m["mean"], yerr=1.96 * m["sem"], marker="o", color="k", capsize=4)
    a.axhline(0, color="grey", lw=.8, ls=":")
    r = prim[(prim.y == y) & main].iloc[0]
    a.set(xlabel="epoch centre (s before awakening)", ylabel=f"{labels[y]}\n(within-awakening, mean ± 95% CI)",
          title=f"slope {r.estimate:+.3f} per 30 s, p_holm = {r.p_holm:.3g}")
piv = chan.pivot(index="channel", columns="y", values="std_estimate")
im = ax[2].imshow(piv.values, cmap="RdBu_r", vmin=-.3, vmax=.3, aspect="auto")
sig = chan.pivot(index="channel", columns="y", values="p_fdr").values < hc["alpha"]
for i, j in zip(*np.where(sig)):
    ax[2].text(j, i, "*", ha="center", va="center")
ax[2].set(xticks=range(piv.shape[1]), xticklabels=piv.columns, yticks=range(piv.shape[0]), yticklabels=piv.index,
          title="Exploratory: standardised slope per 30 s (* FDR < .05)")
plt.colorbar(im, ax=ax[2])
fig.tight_layout(); fig.savefig(figs / "h1_trend.pdf"); fig.savefig(figs / "h1_trend.png", dpi=150)
env.write_manifest(cfg, "04_h1", [tables / f for f in ["h1_primary.csv", "rq3_richness.csv",
                                                       "h1_exploratory_roi.csv", "h1_exploratory_channels.csv"]])

## 5. Post hoc: split by awakening type (exploratory)

**Added after seeing section 1**, so this is an explanation to be tested elsewhere, not a confirmatory result.
Sleep-onset (SO) awakenings wake people shortly after they fall asleep, so their last 45 s may track the
descent into sleep rather than anything that precedes waking. SO awakenings exist only in the `subject` site,
so site is dropped as a covariate within SO.

In [ ]:
roi["awakening_type"] = np.where(roi.tag_class == "SO", "sleep onset (SO)", "other")
split = []
for t, g in roi.groupby("awakening_type"):
    cov = [c for c in hc["covariates"] if c != "site" or g.site.nunique() > 1]
    for y in ["logpow_delta", "logpow_alpha", "logpow_beta", "ap_exponent"]:
        split.append({"awakening_type": t, **stats.fit_trend(g, y, cov)})
split = pd.DataFrame(split)
split["p_fdr"] = stats.fdr_bh(split.p)
split.to_csv(tables / "h1_posthoc_awakening_type.csv", index=False)
print(split[["awakening_type", "y", "std_estimate", "ci_low", "ci_high", "p", "p_fdr", "n_awakenings",
             "n_participants"]].round(4).to_string(index=False))

fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
for a, y in zip(ax, ["logpow_delta", "logpow_alpha", "logpow_beta", "ap_exponent"]):
    for t, c in [("sleep onset (SO)", "#c0504d"), ("other", "#4a6fa5")]:
        d = roi[(roi.awakening_type == t)].dropna(subset=[y]).copy()
        d["centered"] = d[y] - d.groupby("sample_id")[y].transform("mean")
        m = d.groupby("center_s").centered.agg(["mean", "sem"])
        a.errorbar(m.index, m["mean"], yerr=1.96 * m["sem"], marker="o", color=c, capsize=3,
                   label=f"{t} (n={d.sample_id.nunique()})")
    a.axhline(0, color="grey", lw=.8, ls=":")
    a.set(xlabel="epoch centre (s before awakening)", title=f"posterior {y}", ylabel="within-awakening change")
ax[0].legend(fontsize=8)
fig.suptitle("Post hoc (exploratory): pre-awakening change by awakening type")
fig.tight_layout(); fig.savefig(figs / "h1_posthoc_awakening_type.pdf"); fig.savefig(figs / "h1_posthoc_awakening_type.png", dpi=150)